In [10]:
from scipy.stats import norm
import numpy as np
import time
import import_ipynb
#from Geometric_Brownian_Motion import simulate_GBM, simulate_GBM_antithetic
#from Black_Scholes_Closed_Form import european_mc, european_mc_antithetic
from Greeks import pathwise_delta, pathwise_vega, bs_delta, bs_vega
import matplotlib.pyplot as plt
from scipy.stats import qmc

Pathwise Delta: 0.6366 ± 0.0036
BS Delta:       0.6368
Pathwise Vega:  37.2825 ± 0.4668
BS Vega:        37.5240


In [2]:
S0, K, r, sigma, T = 100, 100, 0.05, 0.2, 1.0

In [6]:
def lr_delta(S0, K, r, sigma, T, n_paths, type="call"):
    Z = np.random.normal(0, 1, n_paths)
    S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    
    if type == "call":
        payoff = np.maximum(S_T - K, 0)
    else:
        payoff = np.maximum(K - S_T, 0)
    
    score = Z / (S0 * sigma * np.sqrt(T))
    delta_paths = np.exp(-r*T) * payoff * score
    
    delta = delta_paths.mean()
    se = delta_paths.std(ddof=1) / np.sqrt(n_paths)
    return delta, se

In [7]:
def lr_vega(S0, K, r, sigma, T, n_paths, type="call"):
    Z = np.random.normal(0, 1, n_paths)
    S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    
    if type == "call":
        payoff = np.maximum(S_T - K, 0)
    else:
        payoff = np.maximum(K - S_T, 0)
    
    score = (Z**2 - 1)/sigma - Z*np.sqrt(T)
    vega_paths = np.exp(-r*T) * payoff * score
    
    vega = vega_paths.mean()
    se = vega_paths.std(ddof=1) / np.sqrt(n_paths)
    return vega, se

In [8]:
def fd_delta(S0, K, r, sigma, T, n_paths, type="call", h=0.01):
    seed = 42
    rng1 = np.random.default_rng(seed)
    Z = rng1.standard_normal(n_paths)
    
    S_T_up = (S0+h) * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    S_T_down = (S0-h) * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    
    if type == "call":
        payoff_up = np.maximum(S_T_up - K, 0)
        payoff_down = np.maximum(S_T_down - K, 0)
    else:
        payoff_up = np.maximum(K - S_T_up, 0)
        payoff_down = np.maximum(K - S_T_down, 0)
    
    price_up = np.exp(-r*T) * payoff_up.mean()
    price_down = np.exp(-r*T) * payoff_down.mean()
    
    delta = (price_up - price_down) / (2*h)
    return delta

def fd_vega(S0, K, r, sigma, T, n_paths, type="call", h=0.01):
    seed = 42
    rng1 = np.random.default_rng(seed)
    Z = rng1.standard_normal(n_paths)
    
    S_T_up = S0 * np.exp((r - 0.5*(sigma+h)**2)*T + (sigma+h)*np.sqrt(T)*Z)
    S_T_down = S0 * np.exp((r - 0.5*(sigma-h)**2)*T + (sigma-h)*np.sqrt(T)*Z)
    
    if type == "call":
        payoff_up = np.maximum(S_T_up - K, 0)
        payoff_down = np.maximum(S_T_down - K, 0)
    else:
        payoff_up = np.maximum(K - S_T_up, 0)
        payoff_down = np.maximum(K - S_T_down, 0)
    
    price_up = np.exp(-r*T) * payoff_up.mean()
    price_down = np.exp(-r*T) * payoff_down.mean()
    
    vega = (price_up - price_down) / (2*h)
    return vega

In [11]:
n_paths = 100000

d_pw, se_d_pw = pathwise_delta(S0, K, r, sigma, T, n_paths, "call")
d_lr, se_d_lr = lr_delta(S0, K, r, sigma, T, n_paths, "call")
d_fd = fd_delta(S0, K, r, sigma, T, n_paths, "call")
d_bs = bs_delta(S0, K, r, sigma, T, "call")

v_pw, se_v_pw = pathwise_vega(S0, K, r, sigma, T, n_paths, "call")
v_lr, se_v_lr = lr_vega(S0, K, r, sigma, T, n_paths, "call")
v_fd = fd_vega(S0, K, r, sigma, T, n_paths, "call")
v_bs = bs_vega(S0, K, r, sigma, T)

print("DELTA")
print(f"  BS (analytical):     {d_bs:.4f}")
print(f"  Pathwise:             {d_pw:.4f} ± {1.96*se_d_pw:.4f}")
print(f"  Likelihood ratio:     {d_lr:.4f} ± {1.96*se_d_lr:.4f}")
print(f"  Finite difference:    {d_fd:.4f}")
print()
print("VEGA")
print(f"  BS (analytical):      {v_bs:.4f}")
print(f"  Pathwise:              {v_pw:.4f} ± {1.96*se_v_pw:.4f}")
print(f"  Likelihood ratio:      {v_lr:.4f} ± {1.96*se_v_lr:.4f}")
print(f"  Finite difference:     {v_fd:.4f}")

DELTA
  BS (analytical):     0.6368
  Pathwise:             0.6374 ± 0.0036
  Likelihood ratio:     0.6303 ± 0.0090
  Finite difference:    0.6345

VEGA
  BS (analytical):      37.5240
  Pathwise:              37.6540 ± 0.4668
  Likelihood ratio:      38.2451 ± 1.6834
  Finite difference:     37.4718
